# OpenFold3 Runtime Benchmark

Notebook UI for profiling OpenFold3 runtime behavior across protein lengths.

Workflow:
1. Fill in `pdb_ids_text` and optional `pipeline_trace_pdb_ids`.
2. Adjust `RuntimeConfig` if your OpenFold environment lives somewhere else.
3. Use the controls to preview compositions and launch cold/warm runs.
4. Inspect `case_results.csv`, `events.jsonl`, `samples.jsonl`, and the generated timeline SVGs.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from openfold3_runtime_benchmark import RuntimeConfig, build_notebook_controls, preview_entries, run_runtime_benchmark


In [ ]:
runtime = RuntimeConfig(
    project_dir=PROJECT_ROOT,
    openfold_repo_dir=PROJECT_ROOT / ".." / "openfold-3",
    openfold_prefix=PROJECT_ROOT / ".venv",
    results_dir=PROJECT_ROOT / "openfold3_runtime_benchmark" / "runs" / "_scratch",
    msa_cache_dir=PROJECT_ROOT / ".runtime" / "of3_colabfold_msas",
    triton_cache_dir=Path("/tmp/triton_cache"),
    fixed_msa_tmp_dir=PROJECT_ROOT / ".runtime" / "fixed_msa_tmp",
)

CONFIG = {
    "pdb_ids_text": "1ubq 2crb 5kc1",
    "pipeline_trace_pdb_ids": "1ubq",
    "modes": ("cold", "warm"),
    "sampling_interval_seconds": 1.0,
    "output_root": PROJECT_ROOT / "openfold3_runtime_benchmark" / "runs",
    "runner_yaml": None,
    "max_entries": None,
    "cache_dir": PROJECT_ROOT / "openfold3_runtime_benchmark" / "cache" / "mmcif",
}
state = {}

def get_benchmark_config():
    return dict(CONFIG)

build_notebook_controls(runtime=runtime, config_getter=get_benchmark_config, state=state)


In [ ]:
# Manual preview fallback
preview_entries(
    CONFIG["pdb_ids_text"],
    cache_dir=CONFIG.get("cache_dir"),
    max_entries=CONFIG.get("max_entries"),
)


In [ ]:
# Manual run fallback
result = run_runtime_benchmark(
    runtime=runtime,
    pdb_ids=CONFIG["pdb_ids_text"],
    modes=tuple(CONFIG.get("modes", ("cold", "warm"))),
    sampling_interval_seconds=float(CONFIG.get("sampling_interval_seconds", 1.0)),
    output_root=CONFIG.get("output_root"),
    runner_yaml=CONFIG.get("runner_yaml"),
    max_entries=CONFIG.get("max_entries"),
    cache_dir=CONFIG.get("cache_dir"),
    pipeline_trace_pdb_ids=CONFIG.get("pipeline_trace_pdb_ids"),
)
result.case_results_df
